le pipeline : qu'est-ce qu'il jette, et pourquoi ?

In [1]:

from dvf.config import settings
from dvf.data.build import charger_brut
from dvf.data.clean import agreger_par_mutation, filtrer_ventes, supprimer_prix_manquants

brut = charger_brut(settings.raw_data_dir / "dvf_33_2024.csv.gz")
agrege = brut.pipe(filtrer_ventes).pipe(supprimer_prix_manquants).pipe(agreger_par_mutation)

print(agrege["nb_logements"].value_counts().sort_index().head(8))

nb_logements
0     7753
1    16781
2     2098
3      252
4      121
5       38
6       32
7       10
Name: count, dtype: int64


les logement à un seul bien 16781 mutations
le reste supprimé total de 10304 

nb_logements	| Mutations | 	Part	| Nature

0	| 7 753 |	28,6 %	| Aucun logement dans la vente

1	| 16 781 |	62,0 %	| Conservées

2 et plus |	2 551 |	9,4 %	|Ventes groupées

Sur les 10 304 écartées : 75 % sont des nb_logements == 0, et seulement 25 % sont des ventes groupées.

J'écarte 38 % des mutations. Les trois quarts sont hors périmètre par définition — aucun logement dedans. La vraie limite, ce sont les 9,4 % de ventes groupées, sans prix unitaire exploitable. »

In [7]:
# 1. Quelles ventes ne contiennent aucun logement ?
sans_logement = agrege["nb_logements"] == 0          # série de True/False

print(sans_logement)
ids = set(agrege.loc[sans_logement, "id_mutation"])  # leurs identifiants
print("les ids",ids)
# 2. Revenir aux données AVANT agrégation
verif = brut.pipe(filtrer_ventes).pipe(supprimer_prix_manquants)

# 3. Ne garder que les lignes appartenant à ces ventes
appartient = verif["id_mutation"].isin(ids)          # série de True/False
lignes = verif[appartient]
print("lignes :" , lignes)

# 4. Compter les types de bien qu'on y trouve
lignes["type_local"].value_counts(dropna=False)

0        False
1        False
2        False
3         True
4        False
         ...  
27125    False
27126    False
27127    False
27128    False
27129    False
Name: nb_logements, Length: 27130, dtype: bool
les ids {'2024-386901', '2024-373596', '2024-389746', '2024-391134', '2024-365980', '2024-384871', '2024-381413', '2024-374166', '2024-376290', '2024-387330', '2024-390404', '2024-388864', '2024-379331', '2024-372726', '2024-367006', '2024-381342', '2024-381699', '2024-369595', '2024-380667', '2024-391665', '2024-364294', '2024-380977', '2024-391690', '2024-368534', '2024-377373', '2024-387364', '2024-384496', '2024-393139', '2024-382298', '2024-369204', '2024-377608', '2024-393317', '2024-383235', '2024-381767', '2024-375776', '2024-377745', '2024-389739', '2024-386998', '2024-388951', '2024-372349', '2024-388021', '2024-378185', '2024-381454', '2024-380844', '2024-384688', '2024-386063', '2024-370320', '2024-381871', '2024-390104', '2024-381452', '2024-385420', '2024-366466',

type_local
NaN                                         19818
Local industriel. commercial ou assimilé     1924
Dépendance                                   1665
Name: count, dtype: int64

on supose que les nb_logements == 0 sont des terrains et des garages. et on l'a verifié avec ce code